# 📊 Model Baholash - Amaliy Mashg'ulot

**Maqsad:** Real dataset'lar bilan model evaluation texnikalarini amalda qo'llash

**Mavzular:**
- ✅ Train-Test Split va Cross-Validation
- ✅ Classification va Regression Metrics
- ✅ Overfitting Detection
- ✅ Model Comparison va Tuning

## 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")

---

## 🔬 Exercise 1: Medical Diagnosis (Heart Disease)

**Vazifa:** Yurak kasalligini bashorat qilish va modelni to'g'ri baholash

**Dataset:** Heart Disease UCI

**Qadamlar:**
1. Data yuklash va tahlil qilish
2. Train-Test split
3. Multiple modellarni train qilish
4. Confusion Matrix va Classification Report
5. ROC Curve va AUC
6. Cross-Validation
7. Best model tanlash

In [ ]:
# Load Heart Disease Dataset
from sklearn.datasets import load_breast_cancer  # Similar medical classification dataset

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset Info:")
print(f"  Features: {X.shape[1]}")
print(f"  Samples: {X.shape[0]}")
print(f"  Classes: {np.unique(y)}")
print(f"  Class Distribution:")
print(f"    Class 0 (Malignant): {np.sum(y == 0)} ({np.mean(y == 0)*100:.1f}%)")
print(f"    Class 1 (Benign): {np.sum(y == 1)} ({np.mean(y == 1)*100:.1f}%)")

X.head()

### 1️⃣ Train-Test Split va Preprocessing

In [ ]:
# TODO: Train-test split qiling (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO: StandardScaler bilan scale qiling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data split and scaled!")
print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

### 2️⃣ Train Multiple Models

In [ ]:
# TODO: 4 ta model yarating: Logistic Regression, Decision Tree, Random Forest, KNN
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# Train va evaluate
results = []

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # TODO: Metrics hisoblang
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })

df_results = pd.DataFrame(results)
print("\n" + "="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

### 3️⃣ Confusion Matrix Visualization

In [ ]:
# TODO: Best modelni tanlab, confusion matrix chizing
best_model_name = df_results.loc[df_results['F1-Score'].idxmax(), 'Model']
best_model = models[best_model_name]

y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, 
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            ax=axes[0], linewidths=1, linecolor='black')
axes[0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0].set_title(f'{best_model_name}: Confusion Matrix', fontsize=14, fontweight='bold')

# Metrics Bar Chart
best_metrics = df_results[df_results['Model'] == best_model_name].iloc[0]
metrics_plot = best_metrics[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
metrics_plot.plot(kind='bar', ax=axes[1], alpha=0.8, edgecolor='black', linewidth=1.2, color='skyblue')
axes[1].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title(f'{best_model_name}: All Metrics', fontsize=14, fontweight='bold')
axes[1].set_ylim(0.9, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticklabels(metrics_plot.index, rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"\n✅ Best Model: {best_model_name}")
print(f"   F1-Score: {best_metrics['F1-Score']:.4f}")

### 4️⃣ Cross-Validation

In [ ]:
# TODO: Best modelni Stratified K-Fold (k=5) bilan baholang
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_acc = cross_val_score(best_model, X_train_scaled, y_train, 
                                 cv=skf, scoring='accuracy')
cv_scores_f1 = cross_val_score(best_model, X_train_scaled, y_train, 
                                cv=skf, scoring='f1')

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS (5-Fold Stratified)")
print("="*60)
print(f"Model: {best_model_name}")
print(f"  Accuracy: {cv_scores_acc.mean():.4f} ± {cv_scores_acc.std():.4f}")
print(f"  F1-Score: {cv_scores_f1.mean():.4f} ± {cv_scores_f1.std():.4f}")
print("="*60)

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(1, 6)
width = 0.35
ax.bar(x - width/2, cv_scores_acc, width, label='Accuracy', alpha=0.8, edgecolor='black')
ax.bar(x + width/2, cv_scores_f1, width, label='F1-Score', alpha=0.8, edgecolor='black')
ax.axhline(cv_scores_acc.mean(), color='blue', linestyle='--', lw=1.5, label=f'Mean Accuracy: {cv_scores_acc.mean():.3f}')
ax.axhline(cv_scores_f1.mean(), color='orange', linestyle='--', lw=1.5, label=f'Mean F1: {cv_scores_f1.mean():.3f}')
ax.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'{best_model_name}: Cross-Validation Scores', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---

## 🏠 Exercise 2: House Price Prediction (Regression)

**Vazifa:** Uy narxini bashorat qilish va regression metrics bilan baholash

**Dataset:** California Housing

**Qadamlar:**
1. Data yuklash va tahlil
2. Train-Test split
3. Multiple regression modellar
4. MAE, MSE, RMSE, R² hisoblash
5. Overfitting detection
6. Actual vs Predicted plot

In [ ]:
# Load California Housing Dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
y_house = housing.target

print("Dataset Info:")
print(f"  Features: {X_house.shape[1]}")
print(f"  Samples: {X_house.shape[0]}")
print(f"  Target Range: [{y_house.min():.2f}, {y_house.max():.2f}]")
print(f"  Target Mean: {y_house.mean():.2f}")

X_house.head()

### 1️⃣ Train-Test Split va Scaling

In [ ]:
# TODO: Train-test split qiling (test_size=0.2)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# TODO: Scale qiling
scaler_h = StandardScaler()
X_train_h_scaled = scaler_h.fit_transform(X_train_h)
X_test_h_scaled = scaler_h.transform(X_test_h)

print("✅ Data prepared!")
print(f"  Train: {X_train_h.shape}, Test: {X_test_h.shape}")

### 2️⃣ Train Regression Models

In [ ]:
# TODO: 3 ta regression model yarating
models_reg = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

results_reg = []
predictions = {}

for name, model in models_reg.items():
    # Train
    model.fit(X_train_h_scaled, y_train_h)
    
    # Predict
    y_pred_train = model.predict(X_train_h_scaled)
    y_pred_test = model.predict(X_test_h_scaled)
    predictions[name] = y_pred_test
    
    # TODO: Metrics hisoblang (MAE, MSE, RMSE, R²)
    mae = mean_absolute_error(y_test_h, y_pred_test)
    mse = mean_squared_error(y_test_h, y_pred_test)
    rmse = np.sqrt(mse)
    r2_test = r2_score(y_test_h, y_pred_test)
    r2_train = r2_score(y_train_h, y_pred_train)
    
    results_reg.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R² (Test)': r2_test,
        'R² (Train)': r2_train,
        'Overfit Gap': r2_train - r2_test
    })

df_reg = pd.DataFrame(results_reg)
print("\n" + "="*90)
print("REGRESSION MODELS EVALUATION")
print("="*90)
print(df_reg.to_string(index=False))
print("="*90)

### 3️⃣ Actual vs Predicted Visualization

In [ ]:
# TODO: Har bir modelni Actual vs Predicted plot qiling
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    ax.scatter(y_test_h, y_pred, alpha=0.5, s=20, edgecolor='black', linewidth=0.3)
    ax.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
            'r--', lw=2, label='Perfect Prediction')
    
    r2 = df_reg[df_reg['Model'] == name]['R² (Test)'].values[0]
    rmse = df_reg[df_reg['Model'] == name]['RMSE'].values[0]
    
    ax.set_xlabel('Actual Price', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Price', fontsize=11, fontweight='bold')
    ax.set_title(f'{name}\nR²={r2:.3f}, RMSE={rmse:.3f}', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4️⃣ Overfitting Analysis

In [ ]:
# TODO: Train vs Test R² ni taqqoslang
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x = np.arange(len(df_reg))
width = 0.35

ax.bar(x - width/2, df_reg['R² (Train)'], width, label='Train R²', 
       alpha=0.8, edgecolor='black', linewidth=1.2, color='#2ecc71')
ax.bar(x + width/2, df_reg['R² (Test)'], width, label='Test R²', 
       alpha=0.8, edgecolor='black', linewidth=1.2, color='#e74c3c')

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Detection: Train vs Test R²', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_reg['Model'], rotation=45, ha='right')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.8, color='orange', linestyle='--', lw=1.5, label='Good Threshold (0.8)')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("OVERFITTING ANALYSIS")
print("="*60)
for _, row in df_reg.iterrows():
    gap = row['Overfit Gap']
    status = "✅ Good" if gap < 0.05 else "⚠️ Overfitting" if gap < 0.1 else "❌ Severe Overfitting"
    print(f"{row['Model']:20s} | Gap: {gap:.4f} | {status}")
print("="*60)

---

## 🎯 Exercise 3: Challenge - Diabetes Prediction

**Vazifa:** Diabetes datasetini ishlatib, eng yaxshi modelni toping

**Talablar:**
1. Kamida 5 ta model sinab ko'ring
2. Stratified K-Fold CV (k=10) qo'llang
3. Barcha classification metricslarni hisoblang
4. ROC Curve chizing
5. Best modelni hyperparameter tuning bilan yaxshilang
6. Final test setda natijani ko'ring

In [ ]:
# Load Diabetes Dataset (from workspace)
import os

# Check if diabetes.csv exists in workspace
diabetes_path = '/Users/bnutfilloyev/Developer/lesson/diabetes.csv'

if os.path.exists(diabetes_path):
    df_diabetes = pd.read_csv(diabetes_path)
    print("✅ Diabetes dataset loaded from workspace!")
    print(f"  Shape: {df_diabetes.shape}")
    print(f"  Columns: {list(df_diabetes.columns)}")
    df_diabetes.head()
else:
    print("❌ diabetes.csv not found in workspace!")
    print("   Using Pima Indians Diabetes dataset instead...")
    # Alternative: use sklearn diabetes dataset
    from sklearn.datasets import load_diabetes
    diabetes = load_diabetes()
    df_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
    df_diabetes['target'] = (diabetes.target > diabetes.target.mean()).astype(int)
    print(f"  Shape: {df_diabetes.shape}")
    df_diabetes.head()

In [ ]:
# TODO: Bu yerda o'zingiz ishlang! 🚀
# 1. X va y ajratish
# 2. Train-test split
# 3. Scaling
# 4. Kamida 5 ta model train qilish
# 5. Cross-validation
# 6. Metricslar taqqoslash
# 7. ROC Curve
# 8. Best model tanlash

print("\n💡 HINT: Quyidagi modellarni sinab ko'ring:")
print("   - Logistic Regression")
print("   - Decision Tree")
print("   - Random Forest")
print("   - SVM (SVC)")
print("   - KNN")
print("   - Gradient Boosting (bonus!)\n")

---

## 🎓 Summary

**Bizdagi amaliy mashg'ulotda qanday ko'nikmalarni o'rgandik:**

✅ **Classification Evaluation:**
- Train-Test Split va Stratified Split
- Confusion Matrix va Classification Report
- Multiple metricslar (Accuracy, Precision, Recall, F1, AUC)
- ROC Curve visualization
- Cross-Validation (Stratified K-Fold)
- Multiple models comparison

✅ **Regression Evaluation:**
- Regression metrics (MAE, MSE, RMSE, R²)
- Actual vs Predicted plots
- Overfitting detection (Train vs Test R²)
- Residual analysis

✅ **Best Practices:**
- Always use Stratified Split for imbalanced data
- Check multiple metrics, not just accuracy
- Always validate with Cross-Validation
- Monitor overfitting (Train vs Test gap)
- Visualize results for better understanding

---

**🚀 Keyingi Qadamlar:**
1. `homework.md` - Mustaqil vazifalar
2. `evaluation_guide.md` - Tez yordam qo'llanmasi
3. Real project'larda qo'llash!